# 01 · 多源异构数据摄入--- Ingestion Pipeline

## 实验目的

机器人运控知识库的原始资料来自多种不同平台和格式，每种来源的结构、获取方式、噪声类型都不同。本实验验证以下问题：

1. **五类数据源能否被正确提取**：每种来源的提取器能否拿到干净、可用的文本
2. **切分后的 chunk 能否精确还原**：每个 chunk 记录的字符偏移（char_start / char_end）能否从原始文本中精确还原出 chunk_text，为后续「点击来源高亮原文」功能做准备
3. **不同来源的 chunk 长度分布差异**：判断后续检索排序时是否需要长度归一化

## 六类数据源与样本

| 编号 | 来源类型 | 样本 | 获取方式 |
|------|---------|------|----------|
| 1 | 论文 PDF | Learning Agile Robotic Locomotion Skills by Imitating Animals (2020) | 本地文件 |
| 2 | 规则 PDF | 2026年第二十五届全国大学生机器人大赛ROBOCON比赛规则 | 本地文件 |
| 3 | arXiv 论文 | 搜索关键词自动获取，下载 PDF 解析 | arXiv API |
| 4 | GitHub 项目 | SpotMicroESP32（四足机器人硬件项目） | GitHub API |
| 5 | Gitee 项目 | tinymal-spirit40-t（四足机器人项目） | Gitee API |
| 6 | 普通网页 | ROS 2 官方 Concepts 文档 | HTTP + HTML 解析 |

## 每个模块的结构

每种来源各自成一个独立模块，包含：依赖安装 → 提取 → 切分 → 还原校验；
最后统一做跨来源的 chunk 长度分布对比。

## 共用工具：切分函数 + 校验函数

**先运行这个单元格**，后面所有模块都依赖这里定义的函数。

### 切分策略
按「连续两个以上换行 = 空行 = 段落边界」切段落，累积到接近 target_chars（默认 500 字）才切一刀。不从段落中间截断，保证每个 chunk 语义完整。

### 字符偏移的作用
每个 chunk 记录 char_start 和 char_end，即它在原始文本中的起止位置（字符数）。用户点击「查看来源」时，系统用这两个数字定位并高亮对应段落。偏移记错会导致高亮错位，所以切分后必须立即校验。

In [10]:
import re

def split_paragraphs(text):
    # 按空行（连续两个以上换行）切段落
    # end 延伸到下一段落起始，把段间空白纳入当前段落范围，避免高亮缺口
    raw_segments = list(re.finditer(r'(?:(?!\n\n).)+', text, re.DOTALL))
    paragraphs = []
    for i, m in enumerate(raw_segments):
        content = m.group().strip()
        if not content:
            continue
        p_start = m.start()
        p_end = raw_segments[i + 1].start() if i + 1 < len(raw_segments) else len(text)
        paragraphs.append((p_start, p_end, content))
    return paragraphs


def chunk_text(text, doc_id, doc_type, target_chars=500, min_chars=80):
    # 段落感知切分，返回带字符偏移的 chunk 列表
    # target_chars: 每个 chunk 目标字符数
    # min_chars: 低于此值的短尾段落合并进上一个 chunk，避免碎片
    paragraphs = split_paragraphs(text)
    chunks = []
    buf_start = buf_end = None
    chunk_index = 0

    def flush(b_start, b_end):
        nonlocal chunk_index
        if b_start is None:
            return
        chunk_str = text[b_start:b_end]
        if not chunk_str.strip():
            return
        # 短尾且有前一个 chunk 时才合并，否则直接保留（防止第一个 chunk 丢失）
        if len(chunk_str.strip()) < min_chars and chunks:
            prev = chunks[-1]
            prev['char_end'] = b_end
            prev['chunk_text'] = text[prev['char_start']:b_end]
            prev['char_len'] = len(prev['chunk_text'])
            return
        chunks.append({
            'doc_id': doc_id, 'doc_type': doc_type, 'chunk_index': chunk_index,
            'char_start': b_start, 'char_end': b_end,
            'chunk_text': chunk_str, 'char_len': len(chunk_str),
        })
        chunk_index += 1

    for p_start, p_end, _seg in paragraphs:
        if buf_start is None:
            buf_start, buf_end = p_start, p_end
            continue
        if (buf_end - buf_start) >= target_chars:
            flush(buf_start, buf_end)
            buf_start, buf_end = p_start, p_end
        else:
            buf_end = p_end
    flush(buf_start, buf_end)
    return chunks


def chunk_markdown(text, doc_id, doc_type, target_chars=500, min_chars=80):
    # Markdown 专用切分：优先按 1~3 级标题切 section，超长 section 再细切
    # 比通用切分更好地保留 Markdown 语义结构
    sections = re.split(r'(?m)^(?=#{1,3} )', text)
    all_chunks = []
    search_start = 0

    for section in sections:
        if not section.strip():
            search_start += len(section)
            continue
        sec_start = text.find(section, search_start)
        sec_end = sec_start + len(section)
        search_start = sec_end

        if len(section) <= target_chars:
            all_chunks.append({
                'doc_id': doc_id, 'doc_type': doc_type,
                'chunk_index': len(all_chunks),
                'char_start': sec_start, 'char_end': sec_end,
                'chunk_text': section, 'char_len': len(section),
            })
        else:
            # 超长 section 细切，偏移转换为相对原始 text 的绝对位置
            sub_chunks = chunk_text(section, doc_id=doc_id, doc_type=doc_type,
                                    target_chars=target_chars, min_chars=min_chars)
            for sc in sub_chunks:
                sc['char_start'] += sec_start
                sc['char_end'] += sec_start
                sc['chunk_text'] = text[sc['char_start']:sc['char_end']]
                sc['char_len'] = len(sc['chunk_text'])
                sc['chunk_index'] = len(all_chunks)
                all_chunks.append(sc)
    return all_chunks


def validate_chunks(text, chunks):
    # 还原校验：用 char_start/char_end 从原始文本重新切片，与 chunk_text 逐字比对
    # pass=True 表示所有偏移准确，可以继续往下做 Embedding
    mismatches = []
    for c in chunks:
        rebuilt = text[c['char_start']:c['char_end']]
        if rebuilt != c['chunk_text']:
            mismatches.append({
                'chunk_index': c['chunk_index'],
                'stored_preview': c['chunk_text'][:50],
                'rebuilt_preview': rebuilt[:50],
            })
    return {
        'total_chunks': len(chunks),
        'mismatch_count': len(mismatches),
        'mismatches': mismatches,
        'pass': len(mismatches) == 0,
    }


print('共用函数定义完成，后续所有模块可直接调用')

共用函数定义完成，后续所有模块可直接调用


---
## 模块 1：论文 PDF

### 来源说明
样本：Learning Agile Robotic Locomotion Skills by Imitating Animals（2020），存放在 `实验样本/` 文件夹。

### 提取策略
使用 pymupdf（import 名为 fitz）逐页提取纯文本，页间用双换行拼接。提取后进行噪声清洗：去掉孤立页码行、过短的页眉页脚行（< 8 字符）、References 标题之后的参考文献内容（大量 [1] Author 行会污染向量）。

### 切分策略
使用共用的段落感知切分函数，target_chars=500。论文双栏排版可能导致部分段落顺序混乱，这是 pymupdf 纯文本提取的已知局限，当前阶段接受。

### 注意
运行后观察示例 chunk 内容，确认是正常论文正文，而非页眉页脚或参考文献。

### 依赖
- pymupdf：解析 PDF 文件，安装命令 pip install pymupdf

In [11]:
# pymupdf：用于解析 PDF 文件，论文 PDF 和规则 PDF 模块共用此依赖
!pip install pymupdf -q

zsh:1: command not found: pip


In [12]:
import fitz  # pymupdf 的调用名

def extract_pdf_text(pdf_path):
    # 逐页提取纯文本，页间用双换行拼接，形成段落切分的天然边界
    doc = fitz.open(pdf_path)
    pages_text = [page.get_text() for page in doc]
    doc.close()
    return '\n\n'.join(pages_text)


def clean_pdf_text(text):
    # 清洗三类噪声：孤立页码、过短的页眉页脚、参考文献段
    lines = text.split('\n')
    cleaned = []
    in_references = False
    for line in lines:
        stripped = line.strip()
        if re.match(r'^(References|参考文献|REFERENCES)\s*$', stripped):
            in_references = True
        if in_references:
            continue
        if re.match(r'^\d{1,4}$', stripped):  # 孤立页码
            continue
        if 0 < len(stripped) < 8:  # 过短的页眉页脚，但保留空行
            continue
        cleaned.append(line)
    return '\n'.join(cleaned)


# 执行提取
PAPER_PDF = '实验样本/2020 - Learning Agile Robotic Locomotion Skills by Imitating Animals.pdf'
paper_raw = extract_pdf_text(PAPER_PDF)
paper_text = clean_pdf_text(paper_raw)

print(f'原始提取: {len(paper_raw):,} 字符')
print(f'清洗后:   {len(paper_text):,} 字符（清除噪声 {len(paper_raw)-len(paper_text):,} 字符）')
print('\n前 300 字（确认是正常论文正文）：')
print(paper_text[:300])

ModuleNotFoundError: No module named 'fitz'

In [ ]:
# 切分：doc_type='paper' 标记这是论文类文档
paper_chunks = chunk_text(paper_text, doc_id='paper_agile_locomotion_2020', doc_type='paper')
paper_lens = [c['char_len'] for c in paper_chunks]
print(f'切分出 {len(paper_chunks)} 个 chunk')
print(f'chunk 长度：最小 {min(paper_lens)} / 最大 {max(paper_lens)} / 平均 {sum(paper_lens)//len(paper_lens)}')
print('\n示例 chunk[2]（跳过前两个，通常是标题/作者）：')
print(paper_chunks[2]['chunk_text'][:300])

In [ ]:
# 还原校验：pass=True 才能继续往下做 Embedding
paper_check = validate_chunks(paper_text, paper_chunks)
print(f'[论文 PDF] pass={paper_check["pass"]}, 总 chunk={paper_check["total_chunks"]}, mismatch={paper_check["mismatch_count"]}')
if paper_check['mismatches']:
    print('有问题的 chunk：', paper_check['mismatches'])

---
## 模块 2：规则 PDF

### 来源说明
样本：2026年第二十五届全国大学生机器人大赛 ROBOCON 仿生足式机器人挑战赛比赛规则 V2.0，存放在 `实验样本/` 文件夹。

### 提取策略
与论文 PDF 相同，使用 pymupdf 提取纯文本并清洗。比赛规则通常没有参考文献段，但有页眉（赛事名称）和页脚（页码），同样通过 clean_pdf_text 处理。

### 切分策略
规则文档是短句、分条列举的风格，段落比论文短得多，预期平均 chunk 长度更短。使用相同的 target_chars=500，在最后的跨来源对比中会看到差异。

### 注意
规则文档里的「第X条」「X.X」条款编号保留下来，有助于检索时理解内容语境。

### 依赖
- pymupdf：已在模块 1 安装，无需重复

In [ ]:
# 提取：复用模块 1 定义的函数
RULE_PDF = '实验样本/2026年第二十五届全国大学生机器人大赛ROBOCON仿生足式机器人挑战赛比赛规则-V2.0.pdf'
rule_raw = extract_pdf_text(RULE_PDF)
rule_text = clean_pdf_text(rule_raw)

print(f'原始提取: {len(rule_raw):,} 字符')
print(f'清洗后:   {len(rule_text):,} 字符（清除噪声 {len(rule_raw)-len(rule_text):,} 字符）')
print('\n前 300 字：')
print(rule_text[:300])

In [ ]:
# 切分
rule_chunks = chunk_text(rule_text, doc_id='rule_robocon_2026', doc_type='rule')
rule_lens = [c['char_len'] for c in rule_chunks]
print(f'切分出 {len(rule_chunks)} 个 chunk')
print(f'chunk 长度：最小 {min(rule_lens)} / 最大 {max(rule_lens)} / 平均 {sum(rule_lens)//len(rule_lens)}')
print('\n示例 chunk[0]：')
print(rule_chunks[0]['chunk_text'][:300])

In [ ]:
# 还原校验
rule_check = validate_chunks(rule_text, rule_chunks)
print(f'[规则 PDF] pass={rule_check["pass"]}, 总 chunk={rule_check["total_chunks"]}, mismatch={rule_check["mismatch_count"]}')
if rule_check['mismatches']:
    print('有问题的 chunk：', rule_check['mismatches'])

---
## 模块 3：arXiv 论文

### 来源说明
通过 arXiv 官方 API 搜索机器人运动控制论文，自动下载 PDF 并解析。演示查询：legged robot locomotion reinforcement learning，取第一条结果。

### 提取策略
分两步：（1）arXiv API 返回元信息（标题、作者、摘要、arxiv_id、PDF链接）；（2）用 PDF 链接下载全文，pymupdf + 清洗函数解析。元信息将来存入 documents 表用于来源展示，PDF 全文切分后存入 chunks 表用于检索。

### 注意
arXiv API 有速率限制，不要短时间内发送大量请求。部分论文是扫描版 PDF，pymupdf 提取不了文字，当前暂不处理。

### 依赖
- arxiv：arXiv 官方 Python 客户端，安装命令 pip install arxiv
- requests：下载 PDF 文件
- pymupdf：已安装

In [ ]:
# arxiv：封装了 arXiv API 的调用，免去手动拼接 URL 和解析 XML
# requests：下载论文 PDF 文件
!pip install arxiv requests -q

In [ ]:
import arxiv
import requests
import os

# 通过 arXiv API 搜索论文，取第一条结果的元信息
client = arxiv.Client()
search = arxiv.Search(
    query='legged robot locomotion reinforcement learning',
    max_results=1,
    sort_by=arxiv.SortCriterion.Relevance
)
result = next(client.results(search))

# 元信息将来存入数据库 documents 表，用于来源展示
arxiv_meta = {
    'arxiv_id': result.entry_id.split('/')[-1],
    'title': result.title,
    'authors': [a.name for a in result.authors],
    'published': str(result.published.date()),
    'categories': result.categories,
    'pdf_url': result.pdf_url,
}
print('元信息：')
for k, v in arxiv_meta.items():
    print(f'  {k}: {v}')

In [ ]:
# 下载 PDF 并提取文本，复用模块 1 的函数
arxiv_pdf_path = f'实验样本/arxiv_{arxiv_meta["arxiv_id"]}.pdf'
if not os.path.exists(arxiv_pdf_path):
    # 文件不存在才下载，避免重复下载
    r = requests.get(arxiv_meta['pdf_url'], timeout=60)
    with open(arxiv_pdf_path, 'wb') as f:
        f.write(r.content)
    print(f'下载完成：{len(r.content):,} 字节')
else:
    print(f'文件已存在，跳过下载')

arxiv_raw = extract_pdf_text(arxiv_pdf_path)
arxiv_text = clean_pdf_text(arxiv_raw)
print(f'提取: {len(arxiv_raw):,} 字符 → 清洗后: {len(arxiv_text):,} 字符')

In [ ]:
# 切分，doc_id 用 arxiv_id 确保唯一性且可追溯
arxiv_chunks = chunk_text(arxiv_text, doc_id=f'arxiv_{arxiv_meta["arxiv_id"]}', doc_type='arxiv')
arxiv_lens = [c['char_len'] for c in arxiv_chunks]
print(f'切分出 {len(arxiv_chunks)} 个 chunk')
print(f'chunk 长度：最小 {min(arxiv_lens)} / 最大 {max(arxiv_lens)} / 平均 {sum(arxiv_lens)//len(arxiv_lens)}')

In [ ]:
# 还原校验
arxiv_check = validate_chunks(arxiv_text, arxiv_chunks)
print(f'[arXiv] pass={arxiv_check["pass"]}, 总 chunk={arxiv_check["total_chunks"]}, mismatch={arxiv_check["mismatch_count"]}')
if arxiv_check['mismatches']:
    print('有问题的 chunk：', arxiv_check['mismatches'])

---
## 模块 4：GitHub 项目

### 来源说明
样本：SpotMicroESP32（github.com/michaelkubina/SpotMicroESP32），一个基于 ESP32 的四足机器人硬件项目，包含运动学、电路、组装文档。

### 提取策略
通过 GitHub API 递归遍历仓库目录，找出所有 .md 文件，逐个拉取原始内容。跳过代码文件、图片等非文本文件。每个 .md 文件作为独立文档（独立 doc_id），来源更精确：用户可以看到「来自 SpotMicroESP32 的运动学文档」而非仅「来自 SpotMicroESP32」。

### 切分策略
使用 Markdown 专用切分函数 chunk_markdown()，优先按 1~3 级标题切 section，section 超长再细切。

### 注意
GitHub API 未认证时每小时限 60 次请求，小仓库完全够用。递归层数多时需耐心等待。

### 依赖
- requests：已安装

In [ ]:
import time

def fetch_git_md_files(owner, repo, platform='github', path=''):
    # 递归遍历 GitHub 或 Gitee 仓库，返回所有 .md 文件的路径和下载地址
    # platform 参数区分两个平台，逻辑完全相同，只是 API 地址不同
    if platform == 'github':
        api_url = f'https://api.github.com/repos/{owner}/{repo}/contents/{path}'
    else:  # gitee
        api_url = f'https://gitee.com/api/v5/repos/{owner}/{repo}/contents/{path}'

    resp = requests.get(api_url, timeout=30)
    resp.raise_for_status()
    items = resp.json()

    md_files = []
    for item in items:
        if item['type'] == 'file' and item['name'].lower().endswith('.md'):
            if platform == 'github':
                download_url = item['download_url']
            else:
                # Gitee raw 地址格式与 GitHub 不同
                download_url = f'https://gitee.com/{owner}/{repo}/raw/master/{item["path"]}'
            md_files.append({'path': item['path'], 'download_url': download_url})
        elif item['type'] == 'dir':
            time.sleep(0.3)  # 避免请求过于频繁触发限流
            md_files.extend(fetch_git_md_files(owner, repo, platform, item['path']))
    return md_files


print('正在递归遍历 SpotMicroESP32 仓库...')
github_md_files = fetch_git_md_files('michaelkubina', 'SpotMicroESP32', platform='github')
print(f'找到 {len(github_md_files)} 个 .md 文件：')
for f in github_md_files:
    print(f'  {f["path"]}')

In [ ]:
# 每个 .md 文件独立提取和切分，汇总到 github_all_chunks
github_all_chunks = []
github_texts = {}  # 缓存文本，供校验用，避免重复下载

for md_file in github_md_files:
    resp = requests.get(md_file['download_url'], timeout=30)
    md_text = resp.text
    github_texts[md_file['path']] = md_text

    if not md_text.strip():
        continue

    # doc_id 用文件路径生成，斜杠和点替换为下划线
    safe_path = md_file['path'].replace('/', '_').replace('.', '_').lower()
    doc_id = f'github_spotmicro_{safe_path}'
    chunks = chunk_markdown(md_text, doc_id=doc_id, doc_type='github')
    github_all_chunks.extend(chunks)
    print(f'  {md_file["path"]:45s} -> {len(chunks):3d} chunks')

print(f'\nGitHub 合计：{len(github_all_chunks)} 个 chunk，来自 {len(github_md_files)} 个文件')

In [ ]:
# 还原校验：按文件逐个校验，使用缓存的文本
github_mismatch_total = 0
for md_file in github_md_files:
    safe_path = md_file['path'].replace('/', '_').replace('.', '_').lower()
    doc_id = f'github_spotmicro_{safe_path}'
    file_chunks = [c for c in github_all_chunks if c['doc_id'] == doc_id]
    if not file_chunks:
        continue
    check = validate_chunks(github_texts[md_file['path']], file_chunks)
    github_mismatch_total += check['mismatch_count']
    if check['mismatch_count'] > 0:
        print(f'  FAIL {md_file["path"]}: {check["mismatch_count"]} 个 mismatch')

github_lens = [c['char_len'] for c in github_all_chunks]
print(f'[GitHub] 总 mismatch={github_mismatch_total}, pass={github_mismatch_total == 0}')
print(f'[GitHub] chunk 长度：最小 {min(github_lens)} / 最大 {max(github_lens)} / 平均 {sum(github_lens)//len(github_lens)}')

---
## 模块 5：Gitee 项目

### 来源说明
样本：tinymal-spirit40-t（gitee.com/tinymal/tinymal-spirit40-t），一个四足机器人开源项目。

### 提取策略
Gitee 提供与 GitHub 完全对应的 API 接口。fetch_git_md_files() 函数已支持两个平台，传 platform='gitee' 即可。raw 文件地址格式：gitee.com/{owner}/{repo}/raw/master/{path}。

### 切分策略
与 GitHub 相同，使用 Markdown 专用切分函数。

### 注意
Gitee 的默认分支通常是 master（不是 main），raw URL 中需注意。

### 依赖
- requests：已安装
- fetch_git_md_files 函数：已在模块 4 定义

In [ ]:
# 递归遍历 Gitee 仓库，使用模块 4 定义的通用函数，platform='gitee'
print('正在递归遍历 tinymal-spirit40-t 仓库...')
gitee_md_files = fetch_git_md_files('tinymal', 'tinymal-spirit40-t', platform='gitee')
print(f'找到 {len(gitee_md_files)} 个 .md 文件：')
for f in gitee_md_files:
    print(f'  {f["path"]}')

In [ ]:
# 每个 .md 文件独立提取和切分
gitee_all_chunks = []
gitee_texts = {}

for md_file in gitee_md_files:
    resp = requests.get(md_file['download_url'], timeout=30)
    md_text = resp.text
    gitee_texts[md_file['path']] = md_text

    if not md_text.strip():
        continue

    safe_path = md_file['path'].replace('/', '_').replace('.', '_').lower()
    doc_id = f'gitee_spirit40_{safe_path}'
    chunks = chunk_markdown(md_text, doc_id=doc_id, doc_type='gitee')
    gitee_all_chunks.extend(chunks)
    print(f'  {md_file["path"]:45s} -> {len(chunks):3d} chunks')

print(f'\nGitee 合计：{len(gitee_all_chunks)} 个 chunk，来自 {len(gitee_md_files)} 个文件')

In [ ]:
# 还原校验
gitee_mismatch_total = 0
for md_file in gitee_md_files:
    safe_path = md_file['path'].replace('/', '_').replace('.', '_').lower()
    doc_id = f'gitee_spirit40_{safe_path}'
    file_chunks = [c for c in gitee_all_chunks if c['doc_id'] == doc_id]
    if not file_chunks:
        continue
    check = validate_chunks(gitee_texts[md_file['path']], file_chunks)
    gitee_mismatch_total += check['mismatch_count']
    if check['mismatch_count'] > 0:
        print(f'  FAIL {md_file["path"]}: {check["mismatch_count"]} 个 mismatch')

print(f'[Gitee] 总 mismatch={gitee_mismatch_total}, pass={gitee_mismatch_total == 0}')
if gitee_all_chunks:
    gitee_lens = [c['char_len'] for c in gitee_all_chunks]
    print(f'[Gitee] chunk 长度：最小 {min(gitee_lens)} / 最大 {max(gitee_lens)} / 平均 {sum(gitee_lens)//len(gitee_lens)}')

---
## 模块 6：普通网页

### 来源说明
样本：ROS 2 官方文档 Concepts 页面（docs.ros.org/en/humble/Concepts/Basic.html）。ROS 2 是机器人领域最主流的中间件框架，其官方文档是高质量技术参考资料。选此页面原因：内容纯文字、结构清晰、无登录限制。

### 提取策略
用 requests 下载 HTML，用 BeautifulSoup 解析，优先提取 main 标签内容，其次 article，最后退化到 body。去掉 script、style、nav、footer、header 标签，只保留 p、h1-h4、li、pre 等内容标签的文字，段落间用双换行分隔。

### 切分策略
使用通用段落感知切分函数，网页正文已按段落用双换行分隔。

### 注意
不同网站 HTML 结构差异大，此处的提取逻辑针对 ROS 文档站优化。对于大量 JS 渲染的网站，requests 拿不到完整内容，当前不处理。提取后用肉眼检查，确认没有导航栏文字混入。

### 依赖
- beautifulsoup4：HTML 解析工具，安装命令 pip install beautifulsoup4
- lxml：BeautifulSoup 的高速解析器后端，安装命令 pip install lxml

In [ ]:
# beautifulsoup4：解析 HTML，提取指定标签内的文字
# lxml：beautifulsoup4 的解析器后端，比默认 html.parser 更快更准确
!pip install beautifulsoup4 lxml -q

In [ ]:
from bs4 import BeautifulSoup

def extract_web_text(url):
    # 下载 HTML，优先取 main 标签正文，去掉导航栏等噪声
    headers = {'User-Agent': 'Mozilla/5.0'}  # 伪装浏览器，防止被拒绝
    resp = requests.get(url, headers=headers, timeout=30)
    resp.raise_for_status()
    resp.encoding = resp.apparent_encoding  # 自动检测编码，防止中文乱码

    soup = BeautifulSoup(resp.text, 'lxml')

    # 移除非正文标签
    for tag in soup(['script', 'style', 'nav', 'footer', 'header']):
        tag.decompose()

    # 优先取 main，其次 article，最后 body
    content = soup.find('main') or soup.find('article') or soup.find('body')
    if content is None:
        return ''

    # 提取内容标签文字，段落间用双换行分隔，供 split_paragraphs 识别边界
    paragraphs = []
    for elem in content.find_all(['p', 'h1', 'h2', 'h3', 'h4', 'li', 'pre']):
        text = elem.get_text(separator=' ', strip=True)
        if text:
            paragraphs.append(text)
    return '\n\n'.join(paragraphs)


# 提取
WEB_URL = 'https://docs.ros.org/en/humble/Concepts/Basic.html'
web_text = extract_web_text(WEB_URL)

print(f'提取文本长度: {len(web_text):,} 字符')
print('\n前 400 字（确认是 ROS 文档正文，不是导航栏）：')
print(web_text[:400])

In [ ]:
# 切分
web_chunks = chunk_text(web_text, doc_id='web_ros2_concepts_basic', doc_type='web')
web_lens = [c['char_len'] for c in web_chunks]
print(f'切分出 {len(web_chunks)} 个 chunk')
print(f'chunk 长度：最小 {min(web_lens)} / 最大 {max(web_lens)} / 平均 {sum(web_lens)//len(web_lens)}')
print('\n示例 chunk[0]：')
print(web_chunks[0]['chunk_text'][:300])

In [ ]:
# 还原校验
web_check = validate_chunks(web_text, web_chunks)
print(f'[网页] pass={web_check["pass"]}, 总 chunk={web_check["total_chunks"]}, mismatch={web_check["mismatch_count"]}')
if web_check['mismatches']:
    print('有问题的 chunk：', web_check['mismatches'])

---
## 汇总：跨来源 chunk 长度分布对比

对比六类来源的 chunk 平均长度。

**为什么要对比：** 不同来源文档风格不同，chunk 长度差异大时（超过 2 倍），向量检索排序会有偏差——长 chunk 包含更多词汇，在相似度计算上天然占优。如果差异过大，后续 Re-ranking 阶段需要引入长度归一化。

**注意：** 当前只有六个样本，不能下结论，用于建立初步直觉。接入更多真实论文后（10-15 篇），回来重跑这个单元格。

In [ ]:
# 汇总各来源的 chunk 长度统计
summary = [
    ('论文 PDF',  paper_lens),
    ('规则 PDF',  rule_lens),
    ('arXiv',    arxiv_lens),
    ('GitHub',   github_lens),
    ('Gitee',    [c['char_len'] for c in gitee_all_chunks] if gitee_all_chunks else [0]),
    ('网页',     web_lens),
]

print(f'{"来源":<12} {"chunk数":>8} {"最小":>8} {"最大":>8} {"平均":>8}')
print('-' * 50)
avgs = []
for name, lens in summary:
    if not lens or max(lens) == 0:
        print(f'{name:<12} {"(无数据)":>8}')
        continue
    avg = sum(lens) // len(lens)
    avgs.append((name, avg))
    print(f'{name:<12} {len(lens):>8} {min(lens):>8} {max(lens):>8} {avg:>8}')

if len(avgs) >= 2:
    max_avg = max(a for _, a in avgs)
    min_avg = max(1, min(a for _, a in avgs))
    ratio = max_avg / min_avg
    print(f'\n最大/最小平均长度比值：{ratio:.1f}x')
    if ratio > 2:
        print('提示：差异较大，后续 Re-ranking 阶段需关注长度归一化问题')
    else:
        print('提示：差异在可接受范围内，暂不需要特殊处理')

In [ ]:
# 所有来源的还原校验汇总
print('=== 还原校验汇总 ===')
checks = [
    ('论文 PDF',  paper_check),
    ('规则 PDF',  rule_check),
    ('arXiv',    arxiv_check),
    ('网页',     web_check),
]
for name, check in checks:
    status = 'PASS' if check['pass'] else 'FAIL'
    print(f'  {name:<12} {status}  chunk={check["total_chunks"]}, mismatch={check["mismatch_count"]}')

# GitHub 和 Gitee 汇总
print(f'  {"GitHub":<12} {"PASS" if github_mismatch_total == 0 else "FAIL"}  chunk={len(github_all_chunks)}, mismatch={github_mismatch_total}')
print(f'  {"Gitee":<12} {"PASS" if gitee_mismatch_total == 0 else "FAIL"}  chunk={len(gitee_all_chunks)}, mismatch={gitee_mismatch_total}')
print('\n全部 PASS 后可继续进行 Step 2：Embedding 向量化')